<a href="https://colab.research.google.com/github/ipavlopoulos/ndfu/blob/main/ndfu_example.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# nDFU: structured annotator disagreement

The normalized Distance from Unimodality (nDFU) measures whether an ordinal annotation distribution has one shared peak or separated peaks. A low score means the ratings are compatible with a single tendency. A high score means the histogram departs from that shape, often because annotators form poles such as low ratings and high ratings with few ratings in between.

nDFU detects the shape of the disagreement. It does not by itself explain why the poles exist; that requires inspection, subgroup analysis, or downstream modeling.

In [ ]:
#@title Setup
try:
    from ndfu import dfu, pdf
except ImportError:
    import sys
    import subprocess

    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "git+https://github.com/ipavlopoulos/ndfu.git",
    ])
    from ndfu import dfu, pdf

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid", context="notebook")
SCALE = range(1, 6)


def show_distribution(ratings, title, interpretation):
    """Plot an ordinal annotation histogram and report its nDFU score."""
    hist = pdf(ratings, SCALE)
    score = dfu(hist)

    fig, ax = plt.subplots(figsize=(8, 2.8))
    pd.Series(hist, index=list(SCALE)).plot.bar(ax=ax, color="#0f766e", width=0.8)
    ax.set_title(f"{title} (nDFU = {score:.2f})")
    ax.set_xlabel("Ordinal rating")
    ax.set_ylabel("Relative frequency")
    ax.set_ylim(0, max(0.6, hist.max() + 0.1))
    ax.bar_label(ax.containers[0], fmt="%.2f", padding=2, fontsize=9)
    sns.despine(ax=ax, left=True, bottom=True)
    plt.show()
    plt.close(fig)

    print(f"nDFU: {score:.2f}")
    print(interpretation)
    return hist, score


## Example 1: clear poles

Here, half of the annotators choose the lowest rating and half choose the highest rating. The middle of the scale is empty, so the histogram has two separated peaks. nDFU reports maximum departure from unimodality.

In [ ]:
#@title Clear pole-like disagreement
ratings = [1] * 12 + [5] * 12

hist, score = show_distribution(
    ratings,
    "Clear pole-like disagreement",
    "Annotators form two separated poles: low ratings and high ratings.",
)


## Example 2: poles with some middle ratings

This distribution still has two strong sides, but a few annotators choose ratings in the middle. The disagreement remains structured, so nDFU is still high, but it is lower than the perfectly separated case.

In [ ]:
#@title Pole-like disagreement with some middle ratings
ratings = [1] * 10 + [2, 3, 3, 4] + [5] * 10

hist, score = show_distribution(
    ratings,
    "Pole-like disagreement with some middle ratings",
    "The poles remain visible, but middle ratings reduce the distance from unimodality.",
)


## Example 3: one shared tendency

In this case most annotators choose the middle rating, with fewer annotations toward the extremes. The histogram has one peak, so nDFU is zero.

In [ ]:
#@title Unimodal annotation distribution
ratings = [1] * 2 + [2] * 4 + [3] * 12 + [4] * 4 + [5] * 2

hist, score = show_distribution(
    ratings,
    "One shared tendency",
    "The ratings vary around one central peak, so nDFU is zero.",
)


## Takeaway

Use nDFU when you need to distinguish ordinary variation around one judgment from structured disagreement with separated poles. High-nDFU examples are good candidates for qualitative inspection, subgroup analysis, or K+1 modeling, where pole-like items are treated differently instead of being forced into a single majority label.